# 최종 정리본 안내
- 원본 노트북의 흐름은 최대한 유지했습니다.
- **API Key는 제거**했고, `OPENAI_API_KEY` 환경변수로 읽도록만 변경했습니다.

## 환경변수 설정(Windows PowerShell 예시)
```powershell
setx OPENAI_API_KEY "YOUR_OPENAI_API_KEY"
```
설정 후 **새 터미널/주피터 재실행**이 필요할 수 있습니다.


In [ ]:
import os
# OpenAI API Key는 환경변수에서 읽습니다.
# 필요 시 아래 줄을 실행 전 확인하세요.
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY 환경변수를 설정해주세요 (예: setx OPENAI_API_KEY ...)"


# LLM 기반 복지상담 음성데이터 학습을 통한 우울증 조기진단 예측 모델에 관한 연구

## STT 전처리

### 1. 필요 패키지 설치 및 설정

In [ ]:
# 패키지 설치
!pip install git+https://github.com/openai/whisper.git -q
!pip install jiwer tqdm -q


In [ ]:
# 라이브러리 임포트
import os, json
import whisper
import torch
import zipfile
import pandas as pd
from tqdm import tqdm
from jiwer import wer, cer

### 2. Google Drive 연동 및 압축 해제

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # 최초 실행 시 승인 필요

In [ ]:
zip_path = "/content/drive/MyDrive/샘플전처리.zip"

In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/우울_샘플전처리.zip"
extract_dir = "/content/샘플전처리_병합완료"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("✅ 압축 해제 완료:", extract_dir)


In [ ]:
import os

for root, dirs, files in os.walk(extract_dir):
    print(f"📂 {root}")
    for f in files[:3]:
        print(" └─", f)


In [ ]:
for category in ["우울증", "비우울증"]:
    wav_dir = Path(f"/content/샘플전처리/{category}/wav")
    json_dir = Path(f"/content/샘플전처리/{category}/json")

    copied = 0
    for wav_file in wav_dir.glob("*.wav"):
        stem = wav_file.stem
        src_json = json_dir / f"{stem}.json"
        dst_json = wav_dir / f"{stem}.json"
        if src_json.exists():
            shutil.copy(src_json, dst_json)
            copied += 1
    print(f"✅ {category}: {copied}개 복사 완료")


### 3. Whisper 모델 정의

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = whisper.load_model("tiny", device=device)
print(f"✅ Whisper 모델 로딩 완료 ({device})")

### 4. STT 전처리 함수 정의

In [ ]:
def transcribe_and_evaluate_fixed(root_dir, output_list):
    for category in ["우울증", "비우울증"]:
        folder = os.path.join(root_dir, category, "wav")
        if not os.path.exists(folder):
            print(f"❌ 폴더 없음: {folder}")
            continue

        files = [f for f in os.listdir(folder) if f.lower().endswith(".wav")]
        print(f"🎧 {category} 내 .wav 파일 수:", len(files))

        for f in tqdm(files):
            wav_path = os.path.join(folder, f)
            json_path = wav_path.replace(".wav", ".json")
            speaker_id = os.path.splitext(f)[0]
            label = "depression" if category == "우울증" else "non_depression"

            if not os.path.exists(json_path):
                print(f"❗ JSON 누락: {json_path}")
                continue

            try:
                with open(json_path, 'r', encoding='utf-8') as jf:
                    data = json.load(jf)
                gt_text = data.get("inputText", [{}])[0].get("orgtext", "").strip()
            except Exception as e:
                print(f"⚠️ JSON 파싱 실패: {json_path} → {e}")
                continue

            try:
                result = model.transcribe(wav_path, language="ko")
                pred_text = result["text"].strip()
            except Exception as e:
                print(f"⚠️ STT 실패: {wav_path} → {e}")
                pred_text = ""

            wer_score = wer(gt_text, pred_text) if gt_text and pred_text else None
            cer_score = cer(gt_text, pred_text) if gt_text and pred_text else None

            output_list.append({
                "speaker_id": speaker_id,
                "label": label,
                "gt_text": gt_text,
                "pred_text": pred_text,
                "WER": wer_score,
                "CER": cer_score
            })


### 5. 실행 및 CSV 저장

In [ ]:
output_data = []
transcribe_and_evaluate_fixed("/content/샘플전처리", output_data)


In [ ]:
# 결과 저장
df = pd.DataFrame(output_data)

# 컬럼 보장
expected_columns = ['speaker_id', 'label', 'gt_text', 'pred_text', 'WER', 'CER']
for col in expected_columns:
    if col not in df.columns:
        df[col] = ""
df = pd.DataFrame(output_data)
df.to_csv('/content/depression_stt_result.csv', index=False, encoding='utf-8-sig')
print("✅ 전사 결과 + 평가 CSV 저장 완료")



In [ ]:
# CSV 저장
df = pd.DataFrame(output_data)

# 필수 컬럼 보장
expected_columns = ['speaker_id', 'label', 'gt_text', 'pred_text', 'WER', 'CER']
for col in expected_columns:
    if col not in df.columns:
        df[col] = ""
df = df[expected_columns]

df.to_csv('/content/depression_stt_result.csv', index=False, encoding='utf-8-sig')
print("✅ CSV 저장 완료: depression_stt_result.csv")

### 6. speaker_id 단위로 pred_text 병합 및 저장

In [ ]:
df = pd.read_csv('/content/depression_stt_result.csv')
df['pred_text'] = df['pred_text'].fillna('')

grouped = df.groupby('speaker_id').agg({
    'pred_text': ' '.join,
    'label': 'first'
}).reset_index()

grouped.to_csv('/content/speaker_level_pred_text.csv', index=False, encoding='utf-8-sig')
print("✅ speaker 단위 병합 CSV 저장 완료")

## Scikit-LLM Zero-shot GPT classifier로 "우울 vs 비우울" 분류


*   GPT-3.5-turbo vs GPT-4o 성능 비교
*   정확도, F1-score, 혼동 행렬 등 평가 지표 도출


### 1.Scikit-LLM 설치 및 환경 준비

In [ ]:
!pip install -U scikit-llm openai


### 2.OpenAI API 키 설정

In [ ]:
from skllm.config import SKLLMConfig

SKLLMConfig.set_openai_key(os.getenv("OPENAI_API_KEY"))  # set env var OPENAI_API_KEY
SKLLMConfig.set_openai_org("org-ZHsp52Be6dSxx7pQ2RwYGsb2")


### 3.데이터 불러오기

In [ ]:
import pandas as pd

df = pd.read_csv("/content/speaker_level_pred_text.csv")
df = df.dropna(subset=["pred_text", "label"])
df = df[df["label"].isin(["depression", "non_depression"])]  # 혹시 라벨 오염 방지
df = df.reset_index(drop=True)
print(df["label"].value_counts())
df.head()


### 4. GPT-3.5 분류기 실험 (Zero-Shot)

In [ ]:
from skllm.models.gpt.classification.zero_shot import ZeroShotGPTClassifier
from sklearn.metrics import classification_report, confusion_matrix

clf = ZeroShotGPTClassifier(model="gpt-3.5-turbo")
clf.fit(None, ["depression", "non_depression"])  # 라벨 정의

pred = clf.predict(df["pred_text"].tolist())
df["predicted_3.5"] = pred


In [ ]:
#성능평가 출력
print(classification_report(df["label"], df["predicted_3.5"]))
print("혼동 행렬:")
print(confusion_matrix(df["label"], df["predicted_3.5"]))


### 5. GPT-4o 분류기 비교 실험

In [ ]:
clf4o = ZeroShotGPTClassifier(model="gpt-4o")
clf4o.fit(None, ["depression", "non_depression"])

pred_4o = clf4o.predict(df["pred_text"].tolist())
df["predicted_4o"] = pred_4o

#성능평가 출력
print(classification_report(df["label"], df["predicted_4o"]))
print("혼동 행렬 (GPT-4o):")
print(confusion_matrix(df["label"], df["predicted_4o"]))

### 6. 결과 저장

In [ ]:
df.to_csv("/content/gpt_classification_result.csv", index=False, encoding='utf-8-sig')
print("📁 저장 완료: gpt_classification_result.csv")

## 성능 평가

### [1] 전처리 평가 지표 (STT 기반 WER/CER 평균)

In [ ]:
import pandas as pd

df_stt = pd.read_csv("/content/depression_stt_result.csv")

# WER/CER 평균 계산 (None 제외)
wer_avg = df_stt["WER"].dropna().mean()
cer_avg = df_stt["CER"].dropna().mean()

print(f"✅ Whisper 전사 성능\n- 평균 WER: {wer_avg:.4f}\n- 평균 CER: {cer_avg:.4f}")

[2] GPT 분류 성능 요약 테이블 (표 2용)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def summarize_metrics(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label="depression")
    rec = recall_score(y_true, y_pred, pos_label="depression")
    f1 = f1_score(y_true, y_pred, pos_label="depression")
    return {
        "모델": model_name,
        "정확도": f"{acc:.4f}",
        "정밀도": f"{prec:.4f}",
        "재현율": f"{rec:.4f}",
        "F1-score": f"{f1:.4f}"
    }

# 평가 대상
gpt_35 = summarize_metrics(df["label"], df["predicted_3.5"], "GPT-3.5 turbo")
gpt_4o = summarize_metrics(df["label"], df["predicted_4o"], "GPT-4o-mini")

pd.DataFrame([gpt_35, gpt_4o])


### [3] 혼동 행렬 이미지 (표 3용 시각화)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import rcParams
from sklearn.metrics import confusion_matrix
import seaborn as sns

# 강제 지정할 폰트 경로
font_path = "/content/NanumGothic.ttf"
font_prop = fm.FontProperties(fname=font_path)

cm = confusion_matrix(df["label"], df["predicted_3.5"], labels=["depression", "non_depression"])

#혼동행렬 (GPT-3.5 turbo 기준)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["우울", "비우울"], yticklabels=["우울", "비우울"],
            annot_kws={"fontproperties": font_prop},  # 숫자에도 적용
            cbar_kws={"label": "건수"})

plt.xlabel("실제 라벨", fontsize=12, fontproperties=font_prop)
plt.ylabel("예측 라벨", fontsize=12, fontproperties=font_prop)
plt.title("혼동행렬 (GPT-3.5 turbo 기준)", fontsize=14, fontproperties=font_prop)
plt.xticks(fontproperties=font_prop)
plt.yticks(fontproperties=font_prop)

plt.tight_layout()
plt.savefig("/content/confusion_matrix_gpt35_final.png")
plt.show()


In [ ]:
#GPT-4o 혼동행렬

cm = confusion_matrix(df["label"], df["predicted_4o"], labels=["depression", "non_depression"])

# 시각화
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["우울", "비우울"], yticklabels=["우울", "비우울"],
            annot_kws={"fontproperties": font_prop},
            cbar_kws={"label": "건수"})

plt.xlabel("실제 라벨", fontsize=12, fontproperties=font_prop)
plt.ylabel("예측 라벨", fontsize=12, fontproperties=font_prop)
plt.title("혼동행렬 (GPT-4o 기준)", fontsize=14, fontproperties=font_prop)
plt.xticks(fontproperties=font_prop)
plt.yticks(fontproperties=font_prop)

plt.tight_layout()
plt.savefig("/content/confusion_matrix_gpt4o_final.png")
plt.show()